# Week06 Capstone Project - Neural network Ensemble

## Strategy
For Week 6, the policy uses a small **MLP ensemble surrogate** to score local refinements around the strongest region found so far.

The idea is not to jump globally, but to combine:
1. the current best point,
2. the most recent point,
3. a small, human-clean local candidate set on a fixed grid,
4. an ensemble of shallow MLP regressors for mean and uncertainty,
5. a light UCB-style score to keep a little exploration.

This mirrors the Week 6 theme: a slightly richer model than plain GP-UCB, but still constrained enough to avoid overfitting on a small tabular data set.


In [1]:
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week06.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
32,5,1,2,0.670,0.630,NaN,NaN,NaN,NaN,NaN,NaN,-0.011127
33,5,2,2,0.550,0.650,NaN,NaN,NaN,NaN,NaN,NaN,0.277034
34,5,3,3,0.300,0.500,0.400,NaN,NaN,NaN,NaN,NaN,-0.030842
35,5,4,4,0.415,0.385,0.355,0.455,NaN,NaN,NaN,NaN,-0.093355
36,5,5,4,0.920,0.090,0.960,0.970,NaN,NaN,NaN,NaN,2739.141204
37,5,6,5,0.580,0.230,0.680,0.880,0.07,NaN,NaN,NaN,-0.535497
38,5,7,6,0.030,0.240,0.340,0.240,0.46,0.74,NaN,NaN,1.810761
39,5,8,8,0.390,0.090,0.370,0.060,0.79,0.09,0.1,0.08,9.432110


In [3]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d, rows

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)

def round_grid(x, grid=0.01):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)


In [4]:
def build_local_candidates(best_x, latest_x, second_x, d, step):
    candidates = []
    candidates.append(round_grid(best_x))
    candidates.append(round_grid((best_x + latest_x) / 2.0))
    candidates.append(round_grid((2 * best_x + latest_x) / 3.0))
    candidates.append(round_grid((best_x + second_x) / 2.0))
    candidates.append(round_grid(latest_x))

    for i in range(d):
        for sign in (-1.0, 1.0):
            x = best_x.copy()
            x[i] += sign * step
            candidates.append(round_grid(x))

    for i in range(min(d, 3)):
        x = best_x.copy()
        x[i] += step
        candidates.append(round_grid(x))
        x = best_x.copy()
        x[i] -= step
        candidates.append(round_grid(x))

    candidates = np.unique(np.array(candidates), axis=0)
    return candidates


In [5]:
def ensemble_predict(X_train, y_train, X_test, seeds):
    preds = []
    for seed in seeds:
        model = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(24, 12),
                activation='relu',
                solver='adam',
                alpha=1e-3,
                learning_rate_init=1e-2,
                max_iter=3000,
                random_state=seed,
            )
        )
        model.fit(X_train, y_train)
        preds.append(model.predict(X_test))
    preds = np.vstack(preds)
    return preds.mean(axis=0), preds.std(axis=0)


In [6]:
def suggest_week6_point(function_id, history_df):
    X, y, d, rows = load_combined(function_id, history_df)
    latest_x = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)[-1]
    best_idx = int(np.argmax(y))
    second_idx = int(np.argsort(y)[-2])
    best_x = X[best_idx].copy()
    second_x = X[second_idx].copy()

    base_steps = {2: 0.01, 3: 0.06, 4: 0.01, 5: 0.01, 6: 0.02, 8: 0.02}
    step = base_steps.get(d, 0.02)
    candidates = build_local_candidates(best_x, latest_x, second_x, d, step)

    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    keep = min_dist >= 0.005
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    mean_pred, std_pred = ensemble_predict(X, y, candidates, seeds=[101, 102, 103, 104])
    best_dist = np.sqrt(((candidates - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((candidates - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)

    score = mean_pred + 0.35 * std_pred - 0.20 * best_dist - 0.10 * latest_dist + 0.05 * min_dist
    best_candidate_idx = int(np.argmax(score))
    x_best = candidates[best_candidate_idx]

    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'step': step,
        'mean_pred': float(mean_pred[best_candidate_idx]),
        'std_pred': float(std_pred[best_candidate_idx]),
        'min_dist': float(min_dist[best_candidate_idx]),
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    result = suggest_week6_point(function_id, history)
    results.append(result)
    print(f'Function {function_id}: {result["formatted"]}')
    print(f'  step={result["step"]:.3f}, mean={result["mean_pred"]:.6f}, std={result["std_pred"]:.6f}, min_dist={result["min_dist"]:.6f}')
    print(f'  best_anchor={result["best_anchor"]}')
    print(f'  latest_anchor={result["latest_anchor"]}')
    print()
